# 학습·검증·시험 분할 실습

**Train/Validation/Test Split**

모델 학습, 설정 선택, 최종 성능 확인에 서로 다른 데이터를 쓰는 분할 방식.

소재 분야에서 이해하기: 시험 데이터는 최종 보고 전까지 사용하지 않는다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [scikit-learn 교차검증 문서](https://scikit-learn.org/stable/modules/cross_validation.html)

## 1. 세 덩어리로 나누는 이유

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def make_alloy_data(n=240, noise=6.0, seed=0):
    """개념 확인용 합성 데이터. 실제 합금 측정값이 아닙니다.

    x1 소성 온도(600-900 C), x2 유지 시간(0.5-8 h), x3 첨가 원소 비율(0-5 at%),
    x4 측정 노이즈만 담긴 무의미한 변수. y 는 경도(HV) 를 흉내낸 값입니다.
    """
    rng = np.random.default_rng(seed)
    x1 = rng.uniform(600, 900, n)
    x2 = rng.uniform(0.5, 8.0, n)
    x3 = rng.uniform(0.0, 5.0, n)
    x4 = rng.normal(0.0, 1.0, n)
    y = (120 + 0.14 * (x1 - 600) + 9.0 * np.sqrt(x2) + 11.0 * x3
         - 0.9 * x3 ** 2 - 0.004 * (x1 - 750) * x2 + rng.normal(0, noise, n))
    X = np.column_stack([x1, x2, x3, x4])
    return X, y, ['소성온도', '유지시간', '첨가비율', '무관변수']


X, y, FEATURES = make_alloy_data()
print(X.shape, y.shape, FEATURES)
print('경도 평균 %.1f, 표준편차 %.1f' % (y.mean(), y.std()))

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
X_train, X_valid, y_train, y_valid = train_test_split(X_rest, y_rest, test_size=0.25, random_state=0)
print('학습 %d / 검증 %d / 시험 %d' % (len(X_train), len(X_valid), len(X_test)))

best, best_error = None, np.inf
for depth in (2, 4, 6, 10, None):
    model = RandomForestRegressor(n_estimators=200, max_depth=depth, random_state=0).fit(X_train, y_train)
    error = mean_absolute_error(y_valid, model.predict(X_valid))
    print('  max_depth=%-4s 검증 MAE %.2f' % (depth, error))
    if error < best_error:
        best, best_error = depth, error
print('선택 max_depth=%s' % best)

final = RandomForestRegressor(n_estimators=200, max_depth=best, random_state=0).fit(X_rest, y_rest)
print('시험 MAE %.2f (한 번만 사용)' % mean_absolute_error(y_test, final.predict(X_test)))

## 2. 시험 데이터로 설정을 고르면 어떻게 되나

In [ ]:
cheating = min((mean_absolute_error(y_test, RandomForestRegressor(
    n_estimators=200, max_depth=depth, random_state=0).fit(X_train, y_train).predict(X_test)), depth)
    for depth in (2, 4, 6, 10, None))
print('시험 데이터를 보고 고른 최저 MAE %.2f (실제보다 좋아 보이는 값)' % cheating[0])
print('정직한 절차의 시험 MAE %.2f' % mean_absolute_error(y_test, final.predict(X_test)))

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#train-test-split)을 여세요.